# Tratamento dos dados

## Seleção:
- Triagem das imagens, remover aquelas que possam até mesmo confundir o humano

In [ ]:
from pathlib import Path

import cv2

bottle_path = "./dados_selecionados/Garrafa"
bg_path = "./dados_selecionados/Background"

image = cv2.imread(f"{bottle_path}/7.png")

image_gray = cv2.imread(
        f"{bottle_path}/7.png",
        cv2.IMREAD_GRAYSCALE
    )



In [ ]:
print(image.shape)
print(image_gray.shape)

## Deixando em escalas de cinza

In [ ]:
inputs_dir = ["./dados_selecionados/Garrafa/", "./dados_selecionados/Background/"]
outputs_dir = ["./dados_cinza/Garrafa/", "./dados_cinza/Background/"]

for idx, input_dir in enumerate(inputs_dir):
    input_dir = Path(input_dir)
    output_dir = Path(outputs_dir[idx])

    output_dir.mkdir(parents=True, exist_ok=True)

    for image_path in input_dir.iterdir():
        if not image_path.is_file():
            continue

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        output_path = output_dir / f"{image_path.stem}.png"

        success = cv2.imwrite(str(output_path), image)


## Argumentação dos dados

In [ ]:
inputs_dir = ["./dados_cinza/Garrafa/", "./dados_cinza/Background/"]
outputs_dir = ["./dados_aug/Garrafa/", "./dados_aug/Background/"]

for idx, input_dir in enumerate(inputs_dir):
    input_dir = Path(input_dir)
    output_dir = Path(outputs_dir[idx])

    output_dir.mkdir(parents=True, exist_ok=True)

    for image_path in input_dir.iterdir():
        if not image_path.is_file():
            continue

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        #Flip horizontal
        horizontal = cv2.flip(image, 1)

        #Flip vertical
        vertical = cv2.flip(image, 0)

        rotate_90_cc = cv2.rotate(
            image,
            cv2.ROTATE_90_COUNTERCLOCKWISE
        )

        rotate_90_cw = cv2.rotate(
            image,
            cv2.ROTATE_90_CLOCKWISE
        )

        rotate_180 = cv2.rotate(
            image,
            cv2.ROTATE_180
        )

        augmentations = {
        "flip_h": horizontal,
        "flip_v": vertical,
        "rot_90_cc": rotate_90_cc,
        "rot_90_cw": rotate_90_cw,
        "rot_180": rotate_180,
        }

        for suffix, augmented in augmentations.items():
            output_path = output_dir / (
                f"{image_path.stem}_{suffix}.png"
            )

            cv2.imwrite(str(output_path), augmented)

## Redimensionando a imagem

In [ ]:
inputs_dir = ["./dados_aug/Garrafa/", "./dados_aug/Background/"]
outputs_dir = ["./dados_redm/Garrafa/", "./dados_redm/Background/"]

target_width = 130
target_height = 130

for idx, input_dir in enumerate(inputs_dir):
    input_dir = Path(input_dir)
    output_dir = Path(outputs_dir[idx])

    output_dir.mkdir(parents=True, exist_ok=True)

    for image_path in input_dir.iterdir():
        if not image_path.is_file():
            continue

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        height, width = image.shape
        scale = min(target_width / width, target_height / height)
        
        new_width = round(width * scale)
        new_height = round(height * scale)

        resized = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA
        )

        result = cv2.copyMakeBorder(
            resized,
            top=(target_height - new_height) // 2,
            bottom=(target_height - new_height + 1) // 2,
            left=(target_width - new_width) // 2,
            right=(target_width - new_width + 1) // 2,
            borderType=cv2.BORDER_CONSTANT,
            value=255
        )

        output_path = output_dir / f"{image_path.stem}.png"

        success = cv2.imwrite(str(output_path), result)